# Phishing Website Detection Using Machine Learning

This notebook classifies website records as **phishing** or **legitimate** using four machine-learning models.

## Dataset
- File: `dataset/dataset_phishing.csv`
- Records: 11,430
- Columns: 89 (URL, 87 numeric features, and status)
- Labels: `phishing` and `legitimate`
- Target mapping: phishing = 1, legitimate = 0

> The URL text is excluded from model training. The target column `status` is converted into numeric labels. Logistic Regression and KNN use feature scaling.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# 2. Load the dataset
DATA_PATH = Path("dataset/dataset_phishing.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("dataset_phishing.csv")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nClass distribution:")
display(df["status"].value_counts())

In [ ]:
# 3. Prepare features and target
df = df.drop_duplicates().copy()

X = df.drop(columns=["url", "status"])
y = (df["status"].str.strip().str.lower() == "phishing").astype(int)

print("Records:", len(df))
print("Number of features:", X.shape[1])
print("Missing values in features:", int(X.isna().sum().sum()))
print("Target mapping: 1 = Phishing, 0 = Legitimate")

In [ ]:
# 4. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

In [ ]:
# 5. Define four machine-learning models
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_models[name] = model
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(4))

In [ ]:
# 6. Compare model performance
ax = comparison[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(11, 6), ylim=(0, 1), rot=15
)
ax.set_title("Comparison of Machine Learning Models")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.tight_layout()
plt.show()

best_model_name = comparison["F1-score"].idxmax()
print("Best model based on F1-score:", best_model_name)

In [ ]:
# 7. Classification report for the best model
best_model = trained_models[best_model_name]
best_predictions = predictions[best_model_name]

print(classification_report(
    y_test, best_predictions,
    labels=[1, 0],
    target_names=["Phishing", "Legitimate"],
    zero_division=0
))

In [ ]:
# 8. Confusion matrix for the best model
cm = confusion_matrix(y_test, best_predictions, labels=[1, 0])
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Phishing", "Legitimate"],
    yticklabels=["Phishing", "Legitimate"]
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# 9. Feature importance (tree-based best model)
tree_model = trained_models["Random Forest"].named_steps["classifier"]
importance = pd.Series(tree_model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(15)

display(top_features.to_frame("Importance").round(4))

plt.figure(figsize=(10, 7))
plt.barh(top_features.index[::-1], top_features.values[::-1])
plt.xlabel("Importance score")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importances - Random Forest")
plt.tight_layout()
plt.show()